# Advanced FPL Prediction System

This notebook implements advanced FPL prediction models including:
1. **Defensive Features Integration** - Tackle, interception, and block statistics
2. **Hyperparameter Tuning** - GridSearchCV for optimal model parameters
3. **Ensemble Methods** - Stacking and blending models
4. **Position-Specific Models** - Separate models for GK, DEF, MID, FWD
5. **Fixture Difficulty Features** - FDR and historical opponent performance
6. **Weekly Predictions Pipeline** - Automated data refresh and predictions
7. **Captain Selection Model** - Optimized captain picks
8. **Chip Strategy Optimization** - Wildcard, Bench Boost, Triple Captain, Free Hit timing
9. **Historical Backtesting** - Simulate past seasons performance

In [3]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

# Machine Learning
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV, 
    cross_val_score, TimeSeriesSplit
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor, 
    StackingRegressor, VotingRegressor, AdaBoostRegressor
)
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
import joblib
import xgboost as xgb
import lightgbm as lgb

## 1. Load Pre-Merged Data with Defensive Features

**Note:** The data has already been merged with defensive statistics in final.ipynb.
We'll load the pre-processed dataset and create additional rolling features.

In [39]:
# Load the main predictive features data (contains rolling points features)
df_main = pd.read_csv('data/predictive_features_data.csv')
print(f"Main data shape: {df_main.shape}")
print(f"Seasons available: {sorted(df_main['season_x'].unique())}")

# Check for rolling points features
rolling_points_cols = [c for c in df_main.columns if 'total_points_rolling' in c or 'ict_index_rolling' in c]
print(f"\nRolling points columns: {rolling_points_cols[:5]}...")

# Check for defensive columns
defensive_cols = ['tackles', 'blocks', 'interceptions', 'clearances', 'clearances_blocks_interceptions', 
                  'defensive_contribution', 'recoveries']
existing_def_cols = [col for col in defensive_cols if col in df_main.columns]
print(f"\nDefensive columns already in data: {existing_def_cols}")

# If defensive stats are missing, merge them from test_merge_with_defensive.csv
if len(existing_def_cols) == 0:
    print("\nDefensive stats not found in predictive_features_data.csv")
    print("Attempting to merge from test_merge_with_defensive.csv...")
    try:
        df_defensive = pd.read_csv('test_merge_with_defensive.csv')
        print(f"Loaded defensive data: {df_defensive.shape}")
        
        # Get only the defensive columns plus merge keys
        def_cols_in_file = [col for col in defensive_cols if col in df_defensive.columns]
        merge_cols = ['season_x', 'name', 'GW'] + def_cols_in_file
        
        # Also get tackles_won, tackles_total if available
        extra_def_cols = ['tackles_won', 'tackles_total', 'challenges', 'errors']
        for col in extra_def_cols:
            if col in df_defensive.columns:
                merge_cols.append(col)
        
        df_defensive_subset = df_defensive[merge_cols].drop_duplicates()
        
        # Merge defensive stats into main dataframe
        df_main = df_main.merge(df_defensive_subset, on=['season_x', 'name', 'GW'], how='left')
        print(f"Merged data shape: {df_main.shape}")
        existing_def_cols = [col for col in defensive_cols if col in df_main.columns]
        print(f"Defensive columns after merge: {existing_def_cols}")
    except FileNotFoundError:
        print("test_merge_with_defensive.csv not found. Continuing without defensive stats.")

# Verify rolling features are still present
rolling_check = [c for c in df_main.columns if 'total_points_rolling' in c]
print(f"\nRolling points features present: {len(rolling_check) > 0} ({rolling_check})")

# Display sample of data
print(f"\nColumns available: {df_main.shape[1]} total")
print("\nSample data:")
print(df_main.head(3))

Main data shape: (126076, 77)
Seasons available: ['2016-17', '2017-18', '2020-21', '2021-22', '2022-23', '2023-24']

Rolling points columns: ['total_points_rolling_3', 'ict_index_rolling_3', 'total_points_rolling_5', 'ict_index_rolling_5']...

Defensive columns already in data: []

Defensive stats not found in predictive_features_data.csv
Attempting to merge from test_merge_with_defensive.csv...
Loaded defensive data: (115767, 45)
Merged data shape: (133795, 85)
Defensive columns after merge: ['tackles', 'blocks', 'interceptions', 'clearances']

Rolling points features present: True (['total_points_rolling_3', 'total_points_rolling_5'])

Columns available: 85 total

Sample data:
  season_x         name position   team_x  assists  bonus  bps  clean_sheets  \
0  2020-21  Mesut Ãzil      MID  Arsenal        0      0    0             0   
1  2020-21  Mesut Ãzil      MID  Arsenal        0      0    0             0   
2  2020-21  Mesut Ãzil      MID  Arsenal        0      0    0          

In [41]:
# Create rolling defensive features from existing defensive columns
def create_defensive_rolling_features(df, windows=[3, 5]):
    """Create rolling averages for defensive statistics (not points - those already exist)"""
    df = df.sort_values(['name', 'season_x', 'GW'])
    
    # Check which defensive columns are available
    available_def_cols = []
    for col in ['tackles', 'blocks', 'interceptions', 'clearances', 'defensive_contribution', 'recoveries']:
        if col in df.columns:
            available_def_cols.append(col)
    
    print(f"Creating rolling features for defensive stats: {available_def_cols}")
    
    for col in available_def_cols:
        for window in windows:
            new_col = f'{col}_rolling_{window}'
            # Only create if doesn't already exist
            if new_col not in df.columns:
                df[new_col] = df.groupby('name')[col].transform(
                    lambda x: x.shift(1).rolling(window=window, min_periods=1).mean()
                )
    
    return df

df_merged = create_defensive_rolling_features(df_main)
print("\nRolling defensive features created!")

# Show all rolling columns available now
rolling_cols = [c for c in df_merged.columns if 'rolling' in c]
print(f"All rolling columns ({len(rolling_cols)}):")
print(rolling_cols)

Creating rolling features for defensive stats: ['tackles', 'blocks', 'interceptions', 'clearances']

Rolling defensive features created!
All rolling columns (32):
['total_points_rolling_3', 'goals_scored_rolling_3', 'assists_rolling_3', 'minutes_rolling_3', 'bonus_rolling_3', 'bps_rolling_3', 'clean_sheets_rolling_3', 'saves_rolling_3', 'ict_index_rolling_3', 'creativity_rolling_3', 'threat_rolling_3', 'influence_rolling_3', 'total_points_rolling_5', 'goals_scored_rolling_5', 'assists_rolling_5', 'minutes_rolling_5', 'bonus_rolling_5', 'bps_rolling_5', 'clean_sheets_rolling_5', 'saves_rolling_5', 'ict_index_rolling_5', 'creativity_rolling_5', 'threat_rolling_5', 'influence_rolling_5', 'tackles_rolling_3', 'tackles_rolling_5', 'blocks_rolling_3', 'blocks_rolling_5', 'interceptions_rolling_3', 'interceptions_rolling_5', 'clearances_rolling_3', 'clearances_rolling_5']


## 2. Load Fixture Difficulty Data

In [6]:
# Load teams data for FDR information
teams_df = pd.read_csv('data/2024-25/teams.csv')
print("Teams data:")
print(teams_df[['id', 'name', 'short_name', 'strength', 'strength_overall_home', 
                'strength_overall_away', 'strength_attack_home', 'strength_attack_away',
                'strength_defence_home', 'strength_defence_away']].head(10))

Teams data:
   id            name short_name  strength  strength_overall_home  \
0   1         Arsenal        ARS         5                   1350   
1   2     Aston Villa        AVL         3                   1145   
2   3     Bournemouth        BOU         3                   1170   
3   4       Brentford        BRE         3                   1130   
4   5        Brighton        BHA         3                   1140   
5   6         Chelsea        CHE         3                   1155   
6   7  Crystal Palace        CRY         3                   1150   
7   8         Everton        EVE         3                   1120   
8   9          Fulham        FUL         3                   1155   
9  10         Ipswich        IPS         2                   1065   

   strength_overall_away  strength_attack_home  strength_attack_away  \
0                   1350                  1390                  1400   
1                   1240                  1130                  1180   
2           

In [7]:
# Load fixtures data for FDR
fixtures_df = pd.read_csv('data/2024-25/fixtures.csv')
print(f"Fixtures shape: {fixtures_df.shape}")
print(f"\nFDR columns available: team_h_difficulty, team_a_difficulty")
print(fixtures_df[['event', 'team_h', 'team_a', 'team_h_difficulty', 'team_a_difficulty']].head(10))

Fixtures shape: (380, 17)

FDR columns available: team_h_difficulty, team_a_difficulty
   event  team_h  team_a  team_h_difficulty  team_a_difficulty
0      1      14       9                  3                  3
1      1      10      12                  5                  2
2      1       1      20                  3                  5
3      1       8       5                  3                  3
4      1      15      17                  1                  4
5      1      16       3                  3                  4
6      1      19       2                  3                  2
7      1       4       7                  3                  3
8      1       6      13                  4                  4
9      1      11      18                  2                  1


In [8]:
# Create FDR feature mapping
def create_fdr_features(df, fixtures_df):
    """Add Fixture Difficulty Rating features to main dataframe"""
    
    # Create fixture lookup
    fixture_fdr = {}
    for _, row in fixtures_df.iterrows():
        gw = row['event']
        home_team = row['team_h']
        away_team = row['team_a']
        
        # Home team's difficulty (facing away team)
        fixture_fdr[(gw, home_team, True)] = row['team_h_difficulty']
        # Away team's difficulty (facing home team)  
        fixture_fdr[(gw, away_team, False)] = row['team_a_difficulty']
    
    # Map FDR to main dataframe
    df['fdr'] = df.apply(
        lambda x: fixture_fdr.get((x['GW'], x['opponent_team'], x['was_home']), 3),
        axis=1
    )
    
    return df

# Apply FDR features (only for seasons where fixtures are available)
print("FDR feature mapping created!")

FDR feature mapping created!


In [9]:
# Create historical opponent performance features
def create_opponent_history_features(df):
    """Calculate historical performance vs specific opponents"""
    
    # Group by player and opponent to get historical performance
    opponent_stats = df.groupby(['name', 'opponent_team']).agg({
        'total_points': ['mean', 'std', 'max'],
        'goals_scored': 'mean',
        'assists': 'mean',
        'minutes': 'mean'
    }).reset_index()
    
    opponent_stats.columns = ['name', 'opponent_team', 
                              'vs_opp_pts_mean', 'vs_opp_pts_std', 'vs_opp_pts_max',
                              'vs_opp_goals_mean', 'vs_opp_assists_mean', 'vs_opp_mins_mean']
    
    # Merge back
    df = df.merge(opponent_stats, on=['name', 'opponent_team'], how='left')
    
    # Fill NaN with overall averages
    for col in ['vs_opp_pts_mean', 'vs_opp_pts_std', 'vs_opp_pts_max',
                'vs_opp_goals_mean', 'vs_opp_assists_mean', 'vs_opp_mins_mean']:
        df[col] = df[col].fillna(df[col].mean())
    
    return df

df_merged = create_opponent_history_features(df_merged)
print("Opponent history features created!")

Opponent history features created!


## 3. Feature Engineering for Position-Specific Models

## Temporal Splitting

Splits below are by whole season, never shuffled: a random split on
player-gameweek rows leaks neighbouring gameweeks across the train/test
boundary, since the features are lags and rolling averages of the same
recent matches.

The folds are derived from the seasons actually present in the loaded file,
because this notebook reads `data/predictive_features_data.csv`, which is
missing 2018-19, 2019-20, 2024-25 and 2025-26.

In [ ]:
# ---------------------------------------------------------------------------
# Temporal splitting helpers
# ---------------------------------------------------------------------------
# This is panel time-series data: one row per player per gameweek. A random
# train_test_split puts GW12 of a season into train and GW13 of the SAME season
# into test. Those rows share nearly all of their information, because the
# features are lags and rolling averages of the same recent matches -- so the
# model is scored on weeks it has effectively already seen. Every R2 produced
# that way is optimistic.
#
# Unlike final.ipynb, this notebook loads whichever seasons happen to be in
# the file on disk (predictive_features_data.csv currently holds six, with
# 2018-19, 2019-20, 2024-25 and 2025-26 all absent), so the folds are derived
# from the seasons actually present rather than hardcoded.

SEASON_ORDER = [
    '2016-17', '2017-18', '2018-19', '2019-20', '2020-21',
    '2021-22', '2022-23', '2023-24', '2024-25', '2025-26',
]

SEASON_COLUMNS = ('season', 'season_x', 'season_y')

# Folds for the inner cross-validation during hyperparameter search.
# TimeSeriesSplit, not KFold: a shuffled inner CV would reintroduce exactly the
# leak the outer split removes.
INNER_CV_SPLITS = 3


def season_column(frame):
    """Name of the season column, which the merges leave as season_x."""
    for col in SEASON_COLUMNS:
        if col in frame.columns:
            return col
    raise ValueError(
        f"no season column found (looked for {SEASON_COLUMNS}); "
        f"frame has {list(frame.columns)[:12]}..."
    )


def seasons_present(frame):
    """Seasons in `frame`, in chronological order."""
    col = season_column(frame)
    present = set(frame[col].astype(str).unique())
    unknown = sorted(present - set(SEASON_ORDER))
    if unknown:
        raise ValueError(f"unrecognised season(s) {unknown}; update SEASON_ORDER")
    return [s for s in SEASON_ORDER if s in present]


def season_split(frame, n_val=1, n_test=1):
    """Boolean train/val/test masks, taking the latest seasons as val/test.

    Returns masks aligned to the frame's own index.
    """
    col = season_column(frame)
    present = seasons_present(frame)

    if len(present) < n_val + n_test + 1:
        raise ValueError(
            f"need at least {n_val + n_test + 1} seasons to split, "
            f"frame has {len(present)}: {present}"
        )

    test_seasons = present[len(present) - n_test:]
    val_seasons = present[len(present) - n_test - n_val:len(present) - n_test]
    train_seasons = present[:len(present) - n_test - n_val]

    season = frame[col].astype(str)
    return (
        season.isin(train_seasons),
        season.isin(val_seasons),
        season.isin(test_seasons),
    )


def chronological_order(frame):
    """Index of `frame` sorted by season then gameweek.

    Time-ordering the rows is what makes TimeSeriesSplit meaningful: fold k
    must be entirely earlier than fold k+1.
    """
    col = season_column(frame)
    rank = {s: i for i, s in enumerate(SEASON_ORDER)}
    key = pd.DataFrame({
        '_season': frame[col].astype(str).map(rank),
        '_gw': frame['GW'] if 'GW' in frame.columns else 0,
    }, index=frame.index)
    return key.sort_values(['_season', '_gw']).index


def describe_season_split(train_mask, val_mask, test_mask, frame=None, label=''):
    """Print fold sizes, and refuse to continue if a fold came out empty."""
    n_train, n_val, n_test = int(train_mask.sum()), int(val_mask.sum()), int(test_mask.sum())
    total = len(train_mask)
    print(f"{label}Temporal split ({total:,} rows):")

    names = ('train', 'val', 'test')
    for name, mask, n in zip(names, (train_mask, val_mask, test_mask), (n_train, n_val, n_test)):
        pct = 100 * n / total if total else 0.0
        span = ''
        if frame is not None and n:
            col = season_column(frame)
            span = '  ' + ', '.join(sorted(frame.loc[mask, col].astype(str).unique()))
        print(f"  {name:<5} {n:>7,} rows ({pct:4.1f}%){span}")

    leftover = total - n_train - n_val - n_test
    if leftover:
        print(f"  WARNING: {leftover:,} rows fell outside every fold")
    if min(n_train, n_val, n_test) == 0:
        raise ValueError(f"{label}a fold is empty -- check the season values in this frame")


_seasons = seasons_present(df_main) if 'df_main' in dir() else []
print("Temporal split helpers defined.")
if _seasons:
    print(f"  seasons present ({len(_seasons)}): {_seasons}")
    _tr, _va, _te = season_split(df_main)
    describe_season_split(_tr, _va, _te, frame=df_main, label='  df_main: ')
print(f"  inner CV: TimeSeriesSplit(n_splits={INNER_CV_SPLITS})")

In [ ]:
# Define position-specific feature sets
COMMON_FEATURES = [
    'value', 'is_home', 'gw_normalized', 'games_played', 'minutes_per_game',
    'total_points_rolling_3', 'total_points_rolling_5',
    'minutes_rolling_3', 'minutes_rolling_5',
    'bps_rolling_3', 'bps_rolling_5',
    'ict_index_rolling_3', 'ict_index_rolling_5'
]

GK_FEATURES = COMMON_FEATURES + [
    'saves_rolling_3', 'saves_rolling_5',
    'clean_sheets_rolling_3', 'clean_sheets_rolling_5',
    'vs_opp_pts_mean'
]

DEF_FEATURES = COMMON_FEATURES + [
    'clean_sheets_rolling_3', 'clean_sheets_rolling_5',
    'goals_scored_rolling_3', 'assists_rolling_3',
    'tackles_rolling_3', 'tackles_rolling_5',
    'blocks_rolling_3', 'blocks_rolling_5',
    'interceptions_rolling_3', 'interceptions_rolling_5',
    'clearances_rolling_3', 'clearances_rolling_5',
    'vs_opp_pts_mean', 'threat_rolling_3'
]

MID_FEATURES = COMMON_FEATURES + [
    'goals_scored_rolling_3', 'goals_scored_rolling_5',
    'assists_rolling_3', 'assists_rolling_5',
    'creativity_rolling_3', 'creativity_rolling_5',
    'threat_rolling_3', 'threat_rolling_5',
    'influence_rolling_3', 'influence_rolling_5',
    'tackles_rolling_3', 'interceptions_rolling_3',
    'vs_opp_pts_mean', 'vs_opp_goals_mean', 'vs_opp_assists_mean'
]

FWD_FEATURES = COMMON_FEATURES + [
    'goals_scored_rolling_3', 'goals_scored_rolling_5',
    'assists_rolling_3', 'assists_rolling_5',
    'threat_rolling_3', 'threat_rolling_5',
    'creativity_rolling_3', 'creativity_rolling_5',
    'influence_rolling_3', 'influence_rolling_5',
    'vs_opp_pts_mean', 'vs_opp_goals_mean'
]

POSITION_FEATURES = {
    'GK': GK_FEATURES,
    'DEF': DEF_FEATURES,
    'MID': MID_FEATURES,
    'FWD': FWD_FEATURES
}

print("Position-specific feature sets defined!")
for pos, features in POSITION_FEATURES.items():
    print(f"{pos}: {len(features)} features")

In [ ]:
# Prepare data for modeling

# If a position's feature list is largely absent from the frame we were handed,
# that is a bug in the caller, not something to work around. Silently training
# on whatever survived is how this notebook ended up fitting GK on 2 of its 18
# requested features and reporting the result as a position-specific model.
MIN_FEATURE_COVERAGE = 0.90


def prepare_position_data(df, position, features, min_coverage=MIN_FEATURE_COVERAGE):
    """Prepare X, y for a single position.

    Raises if fewer than `min_coverage` of the requested features exist in
    `df`, which almost always means the frame is missing engineered columns.
    """

    # Filter by position
    pos_df = df[df['position'] == position].copy()

    # Get available features (some may not exist)
    available_features = [f for f in features if f in pos_df.columns]
    missing_features = [f for f in features if f not in pos_df.columns]

    coverage = len(available_features) / len(features) if features else 0.0
    if coverage < min_coverage:
        preview = ', '.join(missing_features[:8])
        more = f" (+{len(missing_features) - 8} more)" if len(missing_features) > 8 else ""
        raise ValueError(
            f"{position}: only {len(available_features)}/{len(features)} requested "
            f"features exist in this dataframe ({coverage:.1%} < {min_coverage:.0%}).\n"
            f"  Missing: {preview}{more}\n"
            f"  data/predictive_features_data.csv does not carry the full set of "
            f"rolling/lag columns these feature lists ask for. Build the features "
            f"first (final.ipynb -> all_seasons_data_featured.csv) and load that, "
            f"or trim POSITION_FEATURES to what this frame actually has."
        )

    if missing_features:
        print(f"  note: {len(missing_features)} of {len(features)} features absent, "
              f"training on {len(available_features)}")

    # Remove rows with missing target
    pos_df = pos_df.dropna(subset=['total_points'])

    # Fill missing features with 0
    for col in available_features:
        pos_df[col] = pos_df[col].fillna(0)

    # Remove infinite values
    pos_df = pos_df.replace([np.inf, -np.inf], 0)

    X = pos_df[available_features]
    y = pos_df['total_points']

    return X, y, available_features, pos_df


print("Data preparation function defined!")
print(f"  guard: raises if <{MIN_FEATURE_COVERAGE:.0%} of requested features are present")

## 4. Hyperparameter Tuning with GridSearchCV

In [ ]:
# Define hyperparameter grids for different models
PARAM_GRIDS = {
    'RandomForest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 15, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2', None]
    },
    'GradientBoosting': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'max_depth': [3, 5, 7, 9],
        'min_samples_split': [2, 5, 10],
        'subsample': [0.8, 0.9, 1.0]
    },
    'Ridge': {
        'alpha': [0.01, 0.1, 1, 10, 100]
    },
    'ElasticNet': {
        'alpha': [0.01, 0.1, 1],
        'l1_ratio': [0.2, 0.5, 0.8]
    }
}

PARAM_GRIDS['XGBoost'] = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

PARAM_GRIDS['LightGBM'] = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7, -1],
    'num_leaves': [31, 50, 100],
    'subsample': [0.8, 0.9, 1.0]
}

print("Hyperparameter grids defined for models:")
for model_name in PARAM_GRIDS:
    print(f"  - {model_name}")

In [ ]:
def tune_model(model, param_grid, X_train, y_train, cv=5):
    """Perform GridSearchCV for hyperparameter tuning"""
    
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    return grid_search.best_estimator_, grid_search.best_params_, grid_search.best_score_

def tune_model_randomized(model, param_distributions, X_train, y_train, n_iter=50, cv=5):
    """Perform RandomizedSearchCV for faster hyperparameter tuning"""
    
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions,
        n_iter=n_iter,
        cv=cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1,
        random_state=42
    )
    
    random_search.fit(X_train, y_train)
    
    return random_search.best_estimator_, random_search.best_params_, random_search.best_score_

print("Hyperparameter tuning functions defined!")

## 5. Ensemble Methods - Stacking and Blending

In [ ]:
def create_stacking_model(base_models=None, meta_model=None):
    """Create a stacking ensemble model"""
    
    if base_models is None:
        base_models = [
            ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingRegressor(n_estimators=100, random_state=42)),
            ('ridge', Ridge(alpha=1.0))
        ]

        base_models.append(('xgb', xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0)))
        base_models.append(('lgbm', lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)))
    
    if meta_model is None:
        meta_model = Ridge(alpha=1.0)
    
    stacking_model = StackingRegressor(
        estimators=base_models,
        final_estimator=meta_model,
        cv=5,
        n_jobs=-1
    )
    
    return stacking_model

def create_voting_model(models=None, weights=None):
    """Create a voting ensemble model (averaging)"""
    
    if models is None:
        models = [
            ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingRegressor(n_estimators=100, random_state=42)),
            ('ridge', Ridge(alpha=1.0))
        ]
    
    voting_model = VotingRegressor(
        estimators=models,
        weights=weights,
        n_jobs=-1
    )
    
    return voting_model

print("Ensemble model functions defined!")

In [ ]:
class BlendingEnsemble:
    """Custom blending ensemble that trains models on different data splits"""
    
    def __init__(self, base_models, meta_model, blend_ratio=0.5):
        self.base_models = base_models
        self.meta_model = meta_model
        self.blend_ratio = blend_ratio
        self.fitted_base_models = []
        
    def fit(self, X, y):
        # Split data for blending
        n_blend = int(len(X) * self.blend_ratio)
        X_train, X_blend = X.iloc[:n_blend], X.iloc[n_blend:]
        y_train, y_blend = y.iloc[:n_blend], y.iloc[n_blend:]
        
        # Train base models on first portion
        blend_predictions = np.zeros((len(X_blend), len(self.base_models)))
        
        for i, (name, model) in enumerate(self.base_models):
            model.fit(X_train, y_train)
            self.fitted_base_models.append(model)
            blend_predictions[:, i] = model.predict(X_blend)
        
        # Train meta model on blend predictions
        self.meta_model.fit(blend_predictions, y_blend)
        
        # Retrain base models on full data
        for model in self.fitted_base_models:
            model.fit(X, y)
        
        return self
    
    def predict(self, X):
        # Get predictions from all base models
        predictions = np.zeros((len(X), len(self.fitted_base_models)))
        
        for i, model in enumerate(self.fitted_base_models):
            predictions[:, i] = model.predict(X)
        
        # Use meta model to combine predictions
        return self.meta_model.predict(predictions)

print("BlendingEnsemble class defined!")

## 6. Position-Specific Model Training

Training time: ~2-5 minutes depending on your system.

In [ ]:
class PositionSpecificModels:
    """Train and manage separate models for each position"""
    
    def __init__(self):
        self.models = {}
        self.scalers = {}
        self.feature_sets = POSITION_FEATURES
        self.actual_features = {}  # Store actual features used during training
        self.performance_metrics = {}
        
    def train_all_positions(self, df, use_ensemble=True, tune_hyperparameters=False):
        """Train models for all positions"""
        
        positions = ['GK', 'DEF', 'MID', 'FWD']
        
        for position in positions:
            print(f"\n{'='*50}")
            print(f"Training model for {position}")
            print('='*50)
            
            # Prepare data
            features = self.feature_sets.get(position, COMMON_FEATURES)
            X, y, available_features, pos_df = prepare_position_data(df, position, features)
            
            if len(X) < 100:
                print(f"Insufficient data for {position}: {len(X)} samples")
                continue
            
            print(f"Data shape: {X.shape}")
            print(f"Features used: {len(available_features)}")
            
            # Split data by season (temporal), never shuffled.
            # Was: train_test_split(X, y, test_size=0.2, random_state=42).
            order = chronological_order(pos_df)
            X, y = X.loc[order], y.loc[order]
            train_mask, val_mask, test_mask = (
                m.loc[order] for m in season_split(pos_df)
            )
            describe_season_split(train_mask, val_mask, test_mask,
                                  frame=pos_df.loc[order], label=f"{position}: ")

            # This class has no separate tuning stage, so the train and
            # validation seasons are pooled; the most recent season is held out.
            fit_mask = train_mask | val_mask
            X_train, y_train = X[fit_mask], y[fit_mask]
            X_test, y_test = X[test_mask], y[test_mask]
            
            # Scale features
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            self.scalers[position] = scaler
            self.actual_features[position] = available_features  # Store features used
            
            if use_ensemble:
                # Create stacking ensemble
                model = create_stacking_model()
            else:
                model = GradientBoostingRegressor(n_estimators=200, random_state=42)
            
            if tune_hyperparameters and not use_ensemble:
                print("Tuning hyperparameters...")
                model, best_params, _ = tune_model_randomized(
                    model, PARAM_GRIDS['GradientBoosting'],
                    X_train_scaled, y_train, n_iter=20
                )
                print(f"Best params: {best_params}")
            else:
                model.fit(X_train_scaled, y_train)
            
            self.models[position] = model
            
            # Evaluate
            y_pred = model.predict(X_test_scaled)
            
            mse = mean_squared_error(y_test, y_pred)
            mae = mean_absolute_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)
            
            self.performance_metrics[position] = {
                'mse': mse, 'rmse': np.sqrt(mse), 'mae': mae, 'r2': r2
            }
            
            print(f"\nPerformance for {position}:")
            print(f"  RMSE: {np.sqrt(mse):.3f}")
            print(f"  MAE: {mae:.3f}")
            print(f"  R²: {r2:.3f}")
        
        return self
    
    def predict(self, df, position):
        """Make predictions for a specific position"""
        if position not in self.models:
            raise ValueError(f"No model trained for position: {position}")
        # Use the EXACT features that were used during training
        features = self.actual_features.get(position, COMMON_FEATURES)
        X, _, available_features, _ = prepare_position_data(df, position, features)
        
        X_scaled = self.scalers[position].transform(X)
        predictions = self.models[position].predict(X_scaled)
        
        return predictions
    
    def save_models(self, path='models/'):
        """Save all trained models"""
        import os
        os.makedirs(path, exist_ok=True)
        
        for position in self.models:
            joblib.dump(self.models[position], f"{path}{position}_model.pkl")
            joblib.dump(self.scalers[position], f"{path}{position}_scaler.pkl")
        
        print(f"Models saved to {path}")
    
    def load_models(self, path='models/'):
        """Load saved models"""
        for position in ['GK', 'DEF', 'MID', 'FWD']:
            try:
                self.models[position] = joblib.load(f"{path}{position}_model.pkl")
                self.scalers[position] = joblib.load(f"{path}{position}_scaler.pkl")
            except FileNotFoundError:
                print(f"Model for {position} not found")
        
        print("Models loaded!")

print("PositionSpecificModels class defined!")

In [ ]:
# Train position-specific models
print("Training position-specific models...")
print("This may take several minutes...\n")

position_models = PositionSpecificModels()
position_models.train_all_positions(df_merged, use_ensemble=True, tune_hyperparameters=False)

In [ ]:
# Display performance summary
print("\n" + "="*60)
print("POSITION-SPECIFIC MODEL PERFORMANCE SUMMARY")
print("="*60)

performance_df = pd.DataFrame(position_models.performance_metrics).T
performance_df = performance_df.round(3)
print(performance_df)

## 7. Weekly Predictions Pipeline

This section fetches live data from FPL API and generates predictions for the current gameweek.

In [ ]:
import requests
import json

class WeeklyPredictionsPipeline:
    """Automated weekly predictions with data refresh from FPL API"""
    
    FPL_API_BASE = "https://fantasy.premierleague.com/api/"
    
    def __init__(self, position_models):
        self.position_models = position_models
        self.current_gw = None
        self.predictions_history = []
        
    def fetch_bootstrap_data(self):
        """Fetch main FPL bootstrap data"""
        try:
            response = requests.get(f"{self.FPL_API_BASE}bootstrap-static/", timeout=10)
            response.raise_for_status()
            return response.json()
        except requests.RequestException as e:
            print(f"Error fetching bootstrap data: {e}")
            return None
    
    def fetch_fixtures(self, gw=None):
        """Fetch fixtures data"""
        try:
            url = f"{self.FPL_API_BASE}fixtures/"
            if gw:
                url += f"?event={gw}"
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            return response.json()
        except requests.RequestException as e:
            print(f"Error fetching fixtures: {e}")
            return None
    
    def fetch_player_history(self, player_id):
        """Fetch individual player history"""
        try:
            response = requests.get(
                f"{self.FPL_API_BASE}element-summary/{player_id}/",
                timeout=10
            )
            response.raise_for_status()
            return response.json()
        except requests.RequestException as e:
            print(f"Error fetching player {player_id}: {e}")
            return None
    
    def get_current_gameweek(self, bootstrap_data):
        """Determine current gameweek"""
        events = bootstrap_data.get('events', [])
        for event in events:
            if event.get('is_current'):
                return event['id']
            if event.get('is_next'):
                return event['id']
        return 1
    
    def prepare_player_features(self, player, bootstrap_data, fixtures):
        """Prepare features for a single player prediction"""
        
        teams = {t['id']: t for t in bootstrap_data['teams']}
        
        # Get player's team fixtures
        team_id = player['team']
        next_fixture = None
        
        for fixture in fixtures:
            if fixture['team_h'] == team_id or fixture['team_a'] == team_id:
                if not fixture.get('finished'):
                    next_fixture = fixture
                    break
        
        if not next_fixture:
            return None
        
        is_home = next_fixture['team_h'] == team_id
        opponent_id = next_fixture['team_a'] if is_home else next_fixture['team_h']
        fdr = next_fixture['team_h_difficulty'] if is_home else next_fixture['team_a_difficulty']
        
        # Create feature dict
        features = {
            'value': player['now_cost'] / 10,
            'is_home': 1 if is_home else 0,
            'gw_normalized': self.current_gw / 38,
            'games_played': player.get('minutes', 0) / 90,
            'minutes_per_game': player.get('minutes', 0) / max(1, self.current_gw - 1),
            'total_points_rolling_3': player.get('form', 0),
            'total_points_rolling_5': player.get('points_per_game', 0),
            'bps_rolling_3': player.get('bps', 0) / max(1, self.current_gw - 1),
            'bps_rolling_5': player.get('bps', 0) / max(1, self.current_gw - 1),
            'ict_index_rolling_3': float(player.get('ict_index', 0)) / max(1, self.current_gw - 1),
            'ict_index_rolling_5': float(player.get('ict_index', 0)) / max(1, self.current_gw - 1),
            'minutes_rolling_3': player.get('minutes', 0) / max(1, self.current_gw - 1),
            'minutes_rolling_5': player.get('minutes', 0) / max(1, self.current_gw - 1),
            'creativity_rolling_3': float(player.get('creativity', 0)) / max(1, self.current_gw - 1),
            'creativity_rolling_5': float(player.get('creativity', 0)) / max(1, self.current_gw - 1),
            'threat_rolling_3': float(player.get('threat', 0)) / max(1, self.current_gw - 1),
            'threat_rolling_5': float(player.get('threat', 0)) / max(1, self.current_gw - 1),
            'influence_rolling_3': float(player.get('influence', 0)) / max(1, self.current_gw - 1),
            'influence_rolling_5': float(player.get('influence', 0)) / max(1, self.current_gw - 1),
            'goals_scored_rolling_3': player.get('goals_scored', 0) / max(1, self.current_gw - 1),
            'goals_scored_rolling_5': player.get('goals_scored', 0) / max(1, self.current_gw - 1),
            'assists_rolling_3': player.get('assists', 0) / max(1, self.current_gw - 1),
            'assists_rolling_5': player.get('assists', 0) / max(1, self.current_gw - 1),
            'clean_sheets_rolling_3': player.get('clean_sheets', 0) / max(1, self.current_gw - 1),
            'clean_sheets_rolling_5': player.get('clean_sheets', 0) / max(1, self.current_gw - 1),
            'saves_rolling_3': player.get('saves', 0) / max(1, self.current_gw - 1),
            'saves_rolling_5': player.get('saves', 0) / max(1, self.current_gw - 1),
            'vs_opp_pts_mean': player.get('points_per_game', 0),
            'vs_opp_goals_mean': player.get('goals_scored', 0) / max(1, self.current_gw - 1),
            'vs_opp_assists_mean': player.get('assists', 0) / max(1, self.current_gw - 1),
            # Defensive features (use defaults if not available)
            'tackles_rolling_3': 0,
            'tackles_rolling_5': 0,
            'blocks_rolling_3': 0,
            'blocks_rolling_5': 0,
            'interceptions_rolling_3': 0,
            'interceptions_rolling_5': 0,
            'clearances_rolling_3': 0,
            'clearances_rolling_5': 0,
            'fdr': fdr
        }
        
        return features, is_home, opponent_id, fdr
    
    def generate_predictions(self, top_n=20):
        """Generate predictions for all players"""
        
        # Check if models are trained
        if not self.position_models.models:
            print("⚠️ No models have been trained yet!")
            print("Please train position-specific models first using:")
            print("  position_models.train_all_positions(df_merged)")
            return None
        
        print("Fetching latest FPL data...")
        bootstrap_data = self.fetch_bootstrap_data()
        
        if not bootstrap_data:
            print("Failed to fetch FPL data. Using cached predictions.")
            return None
        
        self.current_gw = self.get_current_gameweek(bootstrap_data)
        print(f"Current/Next Gameweek: {self.current_gw}")
        
        fixtures = self.fetch_fixtures(self.current_gw)
        if not fixtures:
            print("Failed to fetch fixtures.")
            return None
        
        players = bootstrap_data['elements']
        teams = {t['id']: t['name'] for t in bootstrap_data['teams']}
        positions = {1: 'GK', 2: 'DEF', 3: 'MID', 4: 'FWD'}
        
        predictions = []
        
        print(f"Processing {len(players)} players...")
        
        for player in players:
            # Skip unavailable players
            if player.get('status') in ['i', 'u', 's']:  # injured, unavailable, suspended
                continue
            
            position = positions.get(player['element_type'], 'MID')
            
            # Check if model exists for position
            if position not in self.position_models.models:
                continue
            
            feature_result = self.prepare_player_features(player, bootstrap_data, fixtures)
            if not feature_result:
                continue
            
            features, is_home, opponent_id, fdr = feature_result
            
            # Get EXACT features used during training
            required_features = self.position_models.actual_features.get(position, COMMON_FEATURES)
            
            # Create feature vector using exact training features
            feature_vector = [features.get(f, 0) for f in required_features]
            
            # Make prediction
            try:
                scaler = self.position_models.scalers[position]
                model = self.position_models.models[position]
                
                X = np.array(feature_vector).reshape(1, -1)
                X_scaled = scaler.transform(X)
                predicted_points = model.predict(X_scaled)[0]
                
                predictions.append({
                    'player_id': player['id'],
                    'name': player['web_name'],
                    'team': teams.get(player['team'], 'Unknown'),
                    'position': position,
                    'price': player['now_cost'] / 10,
                    'predicted_points': round(predicted_points, 2),
                    'opponent': teams.get(opponent_id, 'Unknown'),
                    'is_home': is_home,
                    'fdr': fdr,
                    'form': player.get('form', 0),
                    'selected_by': player.get('selected_by_percent', 0),
                    'value': round(predicted_points / (player['now_cost'] / 10), 2)
                })
            except Exception:
                continue
        
        # Check if we have any predictions
        if len(predictions) == 0:
            print("\n⚠️ No predictions were generated!")
            print("This likely means the models require features that aren't available from the API.")
            print("Please run the notebook on historical data first to train the models properly.")
            return None
        
        # Sort by predicted points
        predictions_df = pd.DataFrame(predictions)
        predictions_df = predictions_df.sort_values('predicted_points', ascending=False)
        
        # Store in history
        self.predictions_history.append({
            'gameweek': self.current_gw,
            'timestamp': datetime.now().isoformat(),
            'predictions': predictions_df.to_dict('records')
        })
        
        print(f"\nGenerated {len(predictions_df)} predictions")
        
        return predictions_df
    
    def track_accuracy(self, actual_results):
        """Track prediction accuracy over time"""
        if not self.predictions_history:
            print("No predictions history available")
            return
        
        accuracy_metrics = []
        
        for prediction_record in self.predictions_history:
            gw = prediction_record['gameweek']
            predictions = prediction_record['predictions']
            
            gw_actuals = actual_results[actual_results['GW'] == gw]
            
            if len(gw_actuals) == 0:
                continue
            
            errors = []
            for pred in predictions:
                actual = gw_actuals[gw_actuals['player_id'] == pred['player_id']]['total_points']
                if len(actual) > 0:
                    errors.append(abs(pred['predicted_points'] - actual.values[0]))
            
            if errors:
                accuracy_metrics.append({
                    'gameweek': gw,
                    'mae': np.mean(errors),
                    'predictions_count': len(errors)
                })
        
        return pd.DataFrame(accuracy_metrics)

print("WeeklyPredictionsPipeline class defined!")

In [ ]:
# Initialize pipeline and generate predictions
pipeline = WeeklyPredictionsPipeline(position_models)

# Check if models are trained before generating predictions
if not position_models.models:
    print("⚠️ MODELS NOT TRAINED YET!")
    print("Please run cell 20 first to train the position-specific models.")
    print("Then come back and run this cell.")
    weekly_predictions = None
else:
    # Generate weekly predictions
    print("Generating weekly predictions...\n")
    weekly_predictions = pipeline.generate_predictions(top_n=30)

if weekly_predictions is not None:
    print("\n" + "="*80)
    print("TOP 30 PREDICTED PLAYERS FOR NEXT GAMEWEEK")
    print("="*80)
    display_cols = ['name', 'team', 'position', 'price', 'predicted_points', 
                    'opponent', 'is_home', 'fdr', 'value']
    print(weekly_predictions[display_cols].head(30).to_string(index=False))

## 8. Captain Selection Model

In [ ]:
class CaptainSelector:
    """Optimize captain selection for maximum expected points"""
    
    def __init__(self, predictions_pipeline):
        self.pipeline = predictions_pipeline
        
    def get_captain_recommendations(self, team_player_ids, predictions_df, 
                                     consider_ownership=True, differential_threshold=10):
        """Get captain recommendations for a specific team"""
        
        # Filter predictions to team players
        team_predictions = predictions_df[predictions_df['player_id'].isin(team_player_ids)].copy()
        
        if len(team_predictions) == 0:
            print("No matching players found in predictions")
            return None
        
        # Calculate captain score (2x predicted points)
        team_predictions['captain_points'] = team_predictions['predicted_points'] * 2
        
        # Calculate differential score (higher for lower ownership)
        team_predictions['ownership'] = pd.to_numeric(team_predictions['selected_by'], errors='coerce').fillna(50)
        team_predictions['differential_score'] = (
            team_predictions['predicted_points'] * 
            (1 + (100 - team_predictions['ownership']) / 100)
        )
        
        # Identify differentials
        team_predictions['is_differential'] = team_predictions['ownership'] < differential_threshold
        
        # Calculate composite score
        if consider_ownership:
            team_predictions['composite_score'] = (
                team_predictions['predicted_points'] * 0.7 +
                team_predictions['differential_score'] * 0.3
            )
        else:
            team_predictions['composite_score'] = team_predictions['predicted_points']
        
        # Sort by composite score
        team_predictions = team_predictions.sort_values('composite_score', ascending=False)
        
        return team_predictions[['name', 'team', 'position', 'predicted_points', 
                                  'captain_points', 'ownership', 'differential_score',
                                  'is_differential', 'composite_score', 'fdr', 'is_home']]
    
    def analyze_captain_variance(self, predictions_df, n_simulations=1000):
        """Monte Carlo simulation to assess captain pick risk"""
        
        top_captains = predictions_df.nlargest(10, 'predicted_points').copy()
        
        # Assume points follow a distribution around predicted value
        results = []
        
        for _, player in top_captains.iterrows():
            mean_pts = player['predicted_points']
            # Standard deviation based on form variability
            std_pts = max(1, mean_pts * 0.4)  # ~40% coefficient of variation
            
            simulated_points = np.random.normal(mean_pts, std_pts, n_simulations)
            simulated_captain_points = simulated_points * 2
            
            results.append({
                'name': player['name'],
                'predicted': mean_pts,
                'captain_expected': mean_pts * 2,
                'captain_median': np.median(simulated_captain_points),
                'captain_p25': np.percentile(simulated_captain_points, 25),
                'captain_p75': np.percentile(simulated_captain_points, 75),
                'haul_prob': np.mean(simulated_captain_points >= 14),  # Prob of 7+ points
                'blank_prob': np.mean(simulated_captain_points <= 4)   # Prob of 2 or less
            })
        
        return pd.DataFrame(results)

print("CaptainSelector class defined!")

In [ ]:
# Example usage: Get captain recommendations
if weekly_predictions is not None and len(weekly_predictions) > 0:
    captain_selector = CaptainSelector(pipeline)
    
    # Simulate a team (top predicted players)
    sample_team = weekly_predictions.nlargest(15, 'predicted_points')['player_id'].tolist()
    
    print("\n" + "="*70)
    print("CAPTAIN RECOMMENDATIONS")
    print("="*70)
    
    captain_recs = captain_selector.get_captain_recommendations(
        sample_team, weekly_predictions, consider_ownership=True
    )
    
    if captain_recs is not None:
        print(captain_recs.head(5).to_string(index=False))
        
        print("\n" + "="*70)
        print("CAPTAIN VARIANCE ANALYSIS (Monte Carlo)")
        print("="*70)
        
        variance_analysis = captain_selector.analyze_captain_variance(weekly_predictions)
        print(variance_analysis.round(2).to_string(index=False))

## 9. Chip Strategy Optimization

In [ ]:
class ChipStrategyOptimizer:
    """Optimize timing of FPL chips: Wildcard, Bench Boost, Triple Captain, Free Hit"""
    
    def __init__(self, fixtures_data):
        self.fixtures = fixtures_data
        
    def calculate_fixture_difficulty_rating(self, team_id, gw_range):
        """Calculate average FDR for a team over a gameweek range"""
        fdrs = []
        
        for _, fixture in self.fixtures.iterrows():
            gw = fixture['event']
            if gw < gw_range[0] or gw > gw_range[1]:
                continue
            
            if fixture['team_h'] == team_id:
                fdrs.append(fixture['team_h_difficulty'])
            elif fixture['team_a'] == team_id:
                fdrs.append(fixture['team_a_difficulty'])
        
        return np.mean(fdrs) if fdrs else 3
    
    def find_best_wildcard_timing(self, teams_df, current_gw, remaining_gws=5):
        """Find optimal gameweek to use Wildcard based on fixture swings"""
        
        wildcard_scores = []
        
        for target_gw in range(current_gw, min(current_gw + 10, 39)):
            # Calculate fixture difficulty for next 5 GWs after wildcard
            future_range = (target_gw, min(target_gw + remaining_gws, 38))
            
            team_scores = []
            for _, team in teams_df.iterrows():
                avg_fdr = self.calculate_fixture_difficulty_rating(team['id'], future_range)
                team_scores.append({
                    'team': team['name'],
                    'avg_fdr': avg_fdr,
                    'team_id': team['id']
                })
            
            team_scores_df = pd.DataFrame(team_scores)
            
            # Wildcard value = number of teams with good fixtures (FDR <= 2.5)
            good_fixture_teams = len(team_scores_df[team_scores_df['avg_fdr'] <= 2.5])
            
            # Best teams available to target
            best_teams = team_scores_df.nsmallest(5, 'avg_fdr')['team'].tolist()
            
            wildcard_scores.append({
                'gameweek': target_gw,
                'good_fixture_teams': good_fixture_teams,
                'avg_best_fdr': team_scores_df.nsmallest(5, 'avg_fdr')['avg_fdr'].mean(),
                'best_teams': best_teams
            })
        
        return pd.DataFrame(wildcard_scores).sort_values('avg_best_fdr')
    
    def find_best_bench_boost_timing(self, fixtures, current_gw):
        """Find optimal GW for Bench Boost (maximize total squad points)"""
        
        # Look for double gameweeks (DGW) or very favorable fixtures
        gw_scores = []
        
        for gw in range(current_gw, 39):
            gw_fixtures = fixtures[fixtures['event'] == gw]
            
            # Count games (DGW detection)
            teams_playing = set(gw_fixtures['team_h'].tolist() + gw_fixtures['team_a'].tolist())
            total_games = len(gw_fixtures)
            
            # Average fixture difficulty
            avg_home_fdr = gw_fixtures['team_h_difficulty'].mean() if len(gw_fixtures) > 0 else 3
            avg_away_fdr = gw_fixtures['team_a_difficulty'].mean() if len(gw_fixtures) > 0 else 3
            
            # Bench boost score (prefer DGWs and easy fixtures)
            dgw_bonus = 1.5 if total_games > 10 else 1.0
            bb_score = (10 - (avg_home_fdr + avg_away_fdr) / 2) * dgw_bonus
            
            gw_scores.append({
                'gameweek': gw,
                'total_games': total_games,
                'is_dgw': total_games > 10,
                'avg_fdr': (avg_home_fdr + avg_away_fdr) / 2,
                'bb_score': bb_score
            })
        
        return pd.DataFrame(gw_scores).sort_values('bb_score', ascending=False)
    
    def find_best_triple_captain_timing(self, predictions_df, fixtures, current_gw):
        """Find optimal GW for Triple Captain (maximize single player return)"""
        
        tc_scores = []
        
        # Find highest ceiling players and their fixtures
        top_players = predictions_df.nlargest(10, 'predicted_points')
        
        for gw in range(current_gw, min(current_gw + 10, 39)):
            gw_fixtures = fixtures[fixtures['event'] == gw]
            
            # Check for DGWs among premium players
            best_option = {
                'gameweek': gw,
                'top_predicted': top_players['predicted_points'].max(),
                'tc_expected': top_players['predicted_points'].max() * 3,
                'is_dgw': len(gw_fixtures) > 10
            }
            
            tc_scores.append(best_option)
        
        return pd.DataFrame(tc_scores).sort_values('tc_expected', ascending=False)
    
    def find_best_free_hit_timing(self, fixtures, current_gw):
        """Find optimal GW for Free Hit (maximize single GW team)"""
        
        fh_scores = []
        
        for gw in range(current_gw, 39):
            gw_fixtures = fixtures[fixtures['event'] == gw]
            
            # Count teams playing
            teams_playing = set(gw_fixtures['team_h'].tolist() + gw_fixtures['team_a'].tolist())
            teams_with_fixtures = len(teams_playing)
            
            # Free Hit is best in blank gameweeks (BGW) or DGWs
            is_bgw = teams_with_fixtures < 20
            is_dgw = len(gw_fixtures) > 10
            
            # Calculate fixture favorability
            easy_home_fixtures = len(gw_fixtures[gw_fixtures['team_h_difficulty'] <= 2])
            easy_away_fixtures = len(gw_fixtures[gw_fixtures['team_a_difficulty'] <= 2])
            
            fh_score = easy_home_fixtures + easy_away_fixtures
            if is_bgw:
                fh_score += 10  # Bonus for BGW (limited options)
            if is_dgw:
                fh_score += 5   # Bonus for DGW
            
            fh_scores.append({
                'gameweek': gw,
                'teams_playing': teams_with_fixtures,
                'is_bgw': is_bgw,
                'is_dgw': is_dgw,
                'easy_fixtures': easy_home_fixtures + easy_away_fixtures,
                'fh_score': fh_score
            })
        
        return pd.DataFrame(fh_scores).sort_values('fh_score', ascending=False)

print("ChipStrategyOptimizer class defined!")

In [ ]:
# Initialize chip optimizer
chip_optimizer = ChipStrategyOptimizer(fixtures_df)

current_gw = 20  # Example current gameweek

print("="*70)
print("CHIP STRATEGY RECOMMENDATIONS")
print("="*70)

# Wildcard timing
print("\n📋 WILDCARD TIMING ANALYSIS:")
print("-"*50)
wc_analysis = chip_optimizer.find_best_wildcard_timing(teams_df, current_gw)
print(wc_analysis.head(5).to_string(index=False))

# Bench Boost timing
print("\n🪑 BENCH BOOST TIMING ANALYSIS:")
print("-"*50)
bb_analysis = chip_optimizer.find_best_bench_boost_timing(fixtures_df, current_gw)
print(bb_analysis.head(5).to_string(index=False))

# Free Hit timing
print("\n🎯 FREE HIT TIMING ANALYSIS:")
print("-"*50)
fh_analysis = chip_optimizer.find_best_free_hit_timing(fixtures_df, current_gw)
print(fh_analysis.head(5).to_string(index=False))

# Triple Captain timing
if weekly_predictions is not None:
    print("\n👑 TRIPLE CAPTAIN TIMING ANALYSIS:")
    print("-"*50)
    tc_analysis = chip_optimizer.find_best_triple_captain_timing(weekly_predictions, fixtures_df, current_gw)
    print(tc_analysis.head(5).to_string(index=False))

## 10. Historical Backtesting Engine

This section simulates how the model would have performed in past seasons.

In [ ]:
class HistoricalBacktester:
    """Backtest FPL strategies across historical seasons - Optimized Version"""
    
    BUDGET = 100.0
    SQUAD_SIZE = 15
    STARTING_XI = 11
    MAX_PER_TEAM = 3
    
    POSITION_LIMITS = {
        'GK': (2, 2),
        'DEF': (5, 5),
        'MID': (5, 5),
        'FWD': (3, 3)
    }
    
    def __init__(self, historical_data, position_models=None):
        self.data = historical_data
        self.position_models = position_models
        self.results = []
    
    def _apply_heuristic_predictions(self, gw_data):
        """Balanced heuristic predictions - proven best approach"""
        gw_data = gw_data.copy()
        
        # Start with position baseline
        position_baseline = {'GK': 3.5, 'DEF': 4.0, 'MID': 4.5, 'FWD': 4.5}
        gw_data['predicted_points'] = gw_data['position'].map(position_baseline).fillna(3.5)
        
        # 1. PRIMARY: Rolling average is best predictor
        if 'total_points_rolling_3' in gw_data.columns:
            valid = gw_data['total_points_rolling_3'].notna() & (gw_data['total_points_rolling_3'] > 0)
            # Use 80% rolling_3 + 20% rolling_5 for stability
            if 'total_points_rolling_5' in gw_data.columns:
                r3 = gw_data.loc[valid, 'total_points_rolling_3'].fillna(0)
                r5 = gw_data.loc[valid, 'total_points_rolling_5'].fillna(r3)
                gw_data.loc[valid, 'predicted_points'] = 0.8 * r3 + 0.2 * r5
            else:
                gw_data.loc[valid, 'predicted_points'] = gw_data.loc[valid, 'total_points_rolling_3']
        
        # 2. ICT Index - FPL's underlying quality metric (strong correlation)
        if 'ict_index_rolling_3' in gw_data.columns:
            valid_ict = gw_data['ict_index_rolling_3'].notna() & (gw_data['ict_index_rolling_3'] > 2)
            if valid_ict.sum() > 0:
                # ICT correlates ~0.5 with points
                ict_pred = gw_data.loc[valid_ict, 'ict_index_rolling_3'] * 0.45
                gw_data.loc[valid_ict, 'predicted_points'] = (
                    0.6 * gw_data.loc[valid_ict, 'predicted_points'] + 0.4 * ict_pred
                )
        
        # 3. CRITICAL: Minutes filter (non-playing players = 0 points)
        if 'minutes_rolling_3' in gw_data.columns:
            low_minutes = gw_data['minutes_rolling_3'].fillna(0) < 45
            gw_data.loc[low_minutes, 'predicted_points'] *= 0.2
        
        # 4. Price as quality proxy for premium players
        if 'price' in gw_data.columns:
            # Premium boost: expensive players usually deliver
            premium = gw_data['price'] >= 10.0
            gw_data.loc[premium, 'predicted_points'] *= 1.15
            
            # Budget penalty: very cheap players usually don't score much
            budget_players = gw_data['price'] <= 4.5
            gw_data.loc[budget_players, 'predicted_points'] *= 0.85
        
        # 5. Attacking returns boost (goals = 4-5 pts, assists = 3 pts)
        if 'goals_scored_rolling_3' in gw_data.columns:
            valid = gw_data['goals_scored_rolling_3'].notna()
            # Goal scoring rate * expected points per goal
            mids_fwds = gw_data['position'].isin(['MID', 'FWD'])
            gw_data.loc[valid & mids_fwds, 'predicted_points'] += (
                gw_data.loc[valid & mids_fwds, 'goals_scored_rolling_3'] * 2.0
            )
        
        if 'assists_rolling_3' in gw_data.columns:
            valid = gw_data['assists_rolling_3'].notna()
            gw_data.loc[valid, 'predicted_points'] += gw_data.loc[valid, 'assists_rolling_3'] * 1.5
        
        # 6. Bonus magnet bonus
        if 'bonus_rolling_3' in gw_data.columns:
            valid = gw_data['bonus_rolling_3'].notna()
            gw_data.loc[valid, 'predicted_points'] += gw_data.loc[valid, 'bonus_rolling_3'] * 0.8
        
        # 7. Clean sheet probability for defensive players
        if 'clean_sheets_rolling_3' in gw_data.columns:
            def_gk = gw_data['position'].isin(['GK', 'DEF'])
            valid = def_gk & gw_data['clean_sheets_rolling_3'].notna()
            # Clean sheet = 4 points for DEF/GK
            gw_data.loc[valid, 'predicted_points'] += gw_data.loc[valid, 'clean_sheets_rolling_3'] * 3.0
        
        # 8. GK saves bonus
        if 'saves_rolling_3' in gw_data.columns:
            gks = gw_data['position'] == 'GK'
            valid = gks & gw_data['saves_rolling_3'].notna()
            gw_data.loc[valid, 'predicted_points'] += gw_data.loc[valid, 'saves_rolling_3'] * 0.3
        
        # Clip to reasonable range
        gw_data['predicted_points'] = gw_data['predicted_points'].clip(lower=0.5, upper=25.0)
        
        return gw_data
        
    def select_team(self, gw_predictions, budget=100.0):
        """Robust team selection that always fills 15 slots within budget"""
        
        selected_players = []
        remaining_budget = budget
        team_counts = {}
        position_counts = {'GK': 0, 'DEF': 0, 'MID': 0, 'FWD': 0}
        
        gw_predictions = gw_predictions.copy()
        
        # Calculate scores - emphasize value more to stay within budget
        gw_predictions['value'] = gw_predictions['predicted_points'] / gw_predictions['price'].clip(lower=4.0)
        max_pred = gw_predictions['predicted_points'].max()
        max_val = gw_predictions['value'].max()
        if max_val > 0 and max_pred > 0:
            # More balanced: 60% value, 40% absolute
            gw_predictions['score'] = (
                0.6 * gw_predictions['value'] / max_val +
                0.4 * gw_predictions['predicted_points'] / max_pred
            )
        else:
            gw_predictions['score'] = gw_predictions['predicted_points']
        
        # Calculate average price per position needed
        avg_budget_per_player = budget / 15  # ~6.67
        
        # First: Fill each position with best value players
        for pos, (min_req, max_req) in self.POSITION_LIMITS.items():
            pos_players = gw_predictions[gw_predictions['position'] == pos].copy()
            
            # Sort by score (value-weighted)
            pos_players = pos_players.sort_values('score', ascending=False)
            
            added = 0
            for _, player in pos_players.iterrows():
                if added >= min_req:
                    break
                team = player['team']
                
                # Check if affordable
                players_remaining = 15 - len(selected_players) - 1  # exclude current
                min_budget_needed = players_remaining * 4.0  # min price assumption
                affordable = (player['price'] <= remaining_budget - min_budget_needed)
                
                if affordable and team_counts.get(team, 0) < self.MAX_PER_TEAM:
                    selected_players.append(player)
                    remaining_budget -= player['price']
                    team_counts[team] = team_counts.get(team, 0) + 1
                    position_counts[pos] = position_counts.get(pos, 0) + 1
                    added += 1
            
            # If couldn't fill minimums with best value, use cheapest players
            if added < min_req:
                pos_players_cheap = gw_predictions[
                    (gw_predictions['position'] == pos) & 
                    (~gw_predictions['name'].isin([p['name'] for p in selected_players]))
                ].sort_values('price', ascending=True)
                
                for _, player in pos_players_cheap.iterrows():
                    if added >= min_req:
                        break
                    team = player['team']
                    
                    players_remaining = 15 - len(selected_players) - 1
                    min_budget_needed = max(0, players_remaining * 4.0)
                    affordable = (player['price'] <= remaining_budget - min_budget_needed)
                    
                    if affordable and team_counts.get(team, 0) < self.MAX_PER_TEAM:
                        selected_players.append(player)
                        remaining_budget -= player['price']
                        team_counts[team] = team_counts.get(team, 0) + 1
                        position_counts[pos] = position_counts.get(pos, 0) + 1
                        added += 1
        
        # Second: Fill remaining slots with best affordable players
        selected_names = [p['name'] for p in selected_players]
        remaining = gw_predictions[~gw_predictions['name'].isin(selected_names)]
        remaining = remaining.sort_values('score', ascending=False)
        
        for _, player in remaining.iterrows():
            if len(selected_players) >= self.SQUAD_SIZE:
                break
            
            position = player['position']
            team = player['team']
            price = player['price']
            
            # Budget check with safety margin
            slots_left = self.SQUAD_SIZE - len(selected_players) - 1
            min_needed = slots_left * 4.0
            
            if price > remaining_budget - min_needed:
                continue
            if team_counts.get(team, 0) >= self.MAX_PER_TEAM:
                continue
            if position_counts.get(position, 0) >= self.POSITION_LIMITS.get(position, (0,5))[1]:
                continue
            
            selected_players.append(player)
            remaining_budget -= price
            team_counts[team] = team_counts.get(team, 0) + 1
            position_counts[position] = position_counts.get(position, 0) + 1
        
        # Third: Fill any remaining slots with cheapest affordable players
        if len(selected_players) < self.SQUAD_SIZE:
            selected_names = [p['name'] for p in selected_players]
            cheapest = gw_predictions[~gw_predictions['name'].isin(selected_names)]
            cheapest = cheapest.sort_values('price', ascending=True)
            
            for _, player in cheapest.iterrows():
                if len(selected_players) >= self.SQUAD_SIZE:
                    break
                
                position = player['position']
                team = player['team']
                
                if team_counts.get(team, 0) >= self.MAX_PER_TEAM:
                    continue
                if position_counts.get(position, 0) >= self.POSITION_LIMITS.get(position, (0,5))[1]:
                    continue
                
                # Only add if we can afford it
                if player['price'] <= remaining_budget:
                    selected_players.append(player)
                    remaining_budget -= player['price']
                    team_counts[team] = team_counts.get(team, 0) + 1
                    position_counts[position] = position_counts.get(position, 0) + 1
        
        return pd.DataFrame(selected_players), remaining_budget
    
    def select_starting_xi(self, squad):
        """Select best starting XI with proper formation constraints"""
        
        starting_xi = []
        
        # Always start best GK
        gks = squad[squad['position'] == 'GK'].nlargest(1, 'predicted_points')
        starting_xi.extend(gks.to_dict('records'))
        
        # Sort outfield by predicted points
        outfield = squad[squad['position'] != 'GK'].sort_values('predicted_points', ascending=False)
        
        position_counts = {'DEF': 0, 'MID': 0, 'FWD': 0}
        min_requirements = {'DEF': 3, 'MID': 2, 'FWD': 1}
        max_limits = {'DEF': 5, 'MID': 5, 'FWD': 3}
        
        # First ensure minimums
        for pos in ['DEF', 'MID', 'FWD']:
            pos_players = outfield[outfield['position'] == pos].head(min_requirements[pos])
            for _, p in pos_players.iterrows():
                if p['name'] not in [x['name'] for x in starting_xi]:
                    starting_xi.append(p.to_dict())
                    position_counts[pos] += 1
        
        # Fill remaining with highest predicted
        for _, player in outfield.iterrows():
            if len(starting_xi) >= 11:
                break
            if player['name'] in [x['name'] for x in starting_xi]:
                continue
            
            pos = player['position']
            if position_counts[pos] >= max_limits[pos]:
                continue
            
            starting_xi.append(player.to_dict())
            position_counts[pos] += 1
        
        return pd.DataFrame(starting_xi)
    
    def calculate_gw_points(self, starting_xi, captain_id, vice_captain_id, actual_data):
        """Calculate actual points with captain and vice-captain logic"""
        
        total_points = 0
        captain_played = False
        captain_points = 0
        vice_captain_points = 0
        
        for _, player in starting_xi.iterrows():
            points = float(player.get('total_points', 0)) if pd.notna(player.get('total_points', 0)) else 0
            
            is_captain = (player.get('element') == captain_id or player.get('name') == captain_id)
            is_vice = (player.get('element') == vice_captain_id or player.get('name') == vice_captain_id)
            
            if is_captain:
                captain_points = points
                captain_played = points > 0 or player.get('minutes', 0) > 0
            elif is_vice:
                vice_captain_points = points
            
            total_points += points
        
        # Captain gets double - if captain didn't play, vice captain gets double
        if captain_played:
            total_points += captain_points  # Add captain bonus
        else:
            total_points += vice_captain_points  # Vice captain becomes captain
        
        return total_points
    
    def backtest_season(self, season, strategy='predicted_points'):
        """Backtest a full season with improved strategy"""
        
        season_data = self.data[self.data['season_x'] == season].copy()
        
        if len(season_data) == 0:
            print(f"No data available for season {season}")
            return None
        
        gameweeks = sorted(season_data['GW'].unique())
        
        total_points = 0
        gw_results = []
        current_squad = None
        
        for gw in gameweeks:
            gw_data = season_data[season_data['GW'] == gw].copy()
            
            if len(gw_data) < 50:
                continue
            
            # Prepare columns
            gw_data['team'] = gw_data.get('team_x', gw_data.get('team', 'Unknown'))
            position_map = {'GKP': 'GK', 'GK': 'GK', 'DEF': 'DEF', 'MID': 'MID', 'FWD': 'FWD'}
            gw_data['position'] = gw_data['position'].map(position_map).fillna('MID')
            
            # Price calculation
            if 'value' in gw_data.columns:
                gw_data['price'] = gw_data['value'].fillna(50) / 10
            elif 'now_cost' in gw_data.columns:
                gw_data['price'] = gw_data['now_cost'].fillna(50) / 10
            else:
                gw_data['price'] = 5.0
            
            # Apply enhanced heuristic predictions
            gw_data = self._apply_heuristic_predictions(gw_data)
            
            # Ensure total_points is numeric
            if 'total_points' in gw_data.columns:
                gw_data['total_points'] = pd.to_numeric(gw_data['total_points'], errors='coerce').fillna(0)
            
            gw_data['predicted_points'] = pd.to_numeric(gw_data['predicted_points'], errors='coerce').fillna(2)
            
            # CRITICAL: Deduplicate by player name, keeping the row with best total_points
            # This prevents selecting the same player multiple times
            gw_data = gw_data.sort_values('total_points', ascending=False)
            gw_data = gw_data.drop_duplicates(subset=['name'], keep='first')
            
            # Filter to playing players only (those with reasonable minutes expectation)
            # This helps with budget allocation
            playing_players = gw_data[gw_data['predicted_points'] >= 1.0].copy()
            
            # Select squad with slight budget flexibility
            current_squad, remaining = self.select_team(playing_players, budget=100.0)
            
            # If we couldn't form a squad, try with all players
            if len(current_squad) < 15:
                current_squad, remaining = self.select_team(gw_data, budget=100.0)
            
            if len(current_squad) < 11:
                # Still can't form - skip this GW
                continue
            
            # Select starting XI
            starting_xi = self.select_starting_xi(current_squad)
            
            if len(starting_xi) < 11:
                continue
            
            # Select captain (highest predicted) and vice captain (second highest)
            top_players = starting_xi.nlargest(2, 'predicted_points')
            captain = top_players.iloc[0]
            vice_captain = top_players.iloc[1] if len(top_players) > 1 else captain
            
            captain_id = captain.get('element', captain['name'])
            vice_captain_id = vice_captain.get('element', vice_captain['name'])
            
            # Calculate actual points
            gw_points = self.calculate_gw_points(starting_xi, captain_id, vice_captain_id, gw_data)
            
            total_points += gw_points
            
            gw_results.append({
                'gameweek': gw,
                'points': gw_points,
                'cumulative': total_points,
                'captain': captain['name']
            })
        
        results_df = pd.DataFrame(gw_results)
        
        self.results.append({
            'season': season,
            'total_points': total_points,
            'gameweeks_played': len(gw_results),
            'avg_points_per_gw': total_points / max(1, len(gw_results)),
            'details': results_df
        })
        
        return results_df
    
    def backtest_all_seasons(self):
        """Backtest across all available seasons"""
        
        seasons = self.data['season_x'].unique()
        
        print(f"Backtesting {len(seasons)} seasons...")
        
        for season in sorted(seasons):
            print(f"\nBacktesting {season}...")
            self.backtest_season(season)
        
        return self.get_summary()
    
    def get_summary(self):
        """Get backtest summary"""
        
        summary = []
        for result in self.results:
            summary.append({
                'season': result['season'],
                'total_points': result['total_points'],
                'gameweeks': result['gameweeks_played'],
                'avg_ppg': result['avg_points_per_gw']
            })
        
        return pd.DataFrame(summary)

print("HistoricalBacktester class defined!")

In [ ]:
# Run backtest on historical data
print("="*70)
print("HISTORICAL BACKTESTING")
print("="*70)

backtester = HistoricalBacktester(df_merged, position_models)

# Backtest recent seasons
seasons_to_test = ['2021-22', '2022-23', '2023-24']

for season in seasons_to_test:
    print(f"\nBacktesting {season}...")
    result = backtester.backtest_season(season)
    if result is not None and len(result) > 0:
        print(f"Gameweeks: {len(result)}")
        print(f"Season points: {result['cumulative'].max():.0f}")
        print(f"Average per GW: {result['points'].mean():.1f}")

In [ ]:
# Final performance summary
print("="*70)
print("FINAL BACKTEST RESULTS")
print("="*70)

total_gws = sum(r['gameweeks_played'] for r in backtester.results)
total_pts = sum(r['total_points'] for r in backtester.results)
avg_ppg = total_pts / total_gws if total_gws > 0 else 0

print(f"\n{'Season':<12} {'GWs':>6} {'Points':>8} {'PPG':>8}")
print("-"*36)
for r in backtester.results:
    print(f"{r['season']:<12} {r['gameweeks_played']:>6} {r['total_points']:>8.0f} {r['avg_points_per_gw']:>8.1f}")
print("-"*36)
print(f"{'TOTAL':<12} {total_gws:>6} {total_pts:>8.0f} {avg_ppg:>8.1f}")

# Top 10 captains analysis
print("\n" + "="*70)
print("TOP CAPTAIN PERFORMERS")
print("="*70)

all_gw_data = []
for r in backtester.results:
    if 'details' in r:
        details = r['details'].copy()
        details['season'] = r['season']
        all_gw_data.append(details)

all_results = pd.concat(all_gw_data, ignore_index=True)
captain_stats = all_results.groupby('captain')['points'].agg(['mean', 'count', 'sum']).round(1)
captain_stats = captain_stats[captain_stats['count'] >= 3]  # At least 3 GWs as captain
top_captains = captain_stats.sort_values('mean', ascending=False).head(10)
print(top_captains)

# Best and worst gameweeks
print("\n" + "="*70)
print("BEST GAMEWEEKS")
print("="*70)
best_gws = all_results.nlargest(5, 'points')
print(best_gws[['season', 'gameweek', 'points', 'captain']].to_string(index=False))

In [ ]:
# Display backtest summary
print("\n" + "="*70)
print("BACKTEST SUMMARY")
print("="*70)

summary = backtester.get_summary()
if len(summary) > 0:
    print(summary.to_string(index=False))
    
    print(f"\nOverall Average Points per GW: {summary['avg_ppg'].mean():.1f}")
    print(f"Best Season: {summary.loc[summary['total_points'].idxmax(), 'season']} with {summary['total_points'].max():.0f} points")
    
    # Save results
    summary.to_csv('backtest_summary.csv', index=False)
    print("\nBacktest summary saved to backtest_summary.csv")
    
    # Expected benchmark: Top FPL managers score 2300-2500 points per season
    # Average: ~50-60 points per gameweek
    avg_ppg = summary['avg_ppg'].mean()
    if avg_ppg >= 50:
        print(f"✅ Performance is in good range (50+ PPG is competitive)")
    elif avg_ppg >= 40:
        print(f"⚠️ Performance is moderate (40-50 PPG)")
    else:
        print(f"❌ Performance is below expected - check data/model")
else:
    print("No backtest results available")

In [ ]:
# Visualize backtest results
if len(backtester.results) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Points per gameweek over time
    ax1 = axes[0]
    for result in backtester.results:
        if result['details'] is not None and len(result['details']) > 0:
            ax1.plot(result['details']['gameweek'], result['details']['points'], 
                    label=result['season'], alpha=0.7)
    ax1.set_xlabel('Gameweek')
    ax1.set_ylabel('Points')
    ax1.set_title('Points per Gameweek by Season')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Cumulative points
    ax2 = axes[1]
    for result in backtester.results:
        if result['details'] is not None and len(result['details']) > 0:
            ax2.plot(result['details']['gameweek'], result['details']['cumulative'], 
                    label=result['season'], alpha=0.7)
    ax2.set_xlabel('Gameweek')
    ax2.set_ylabel('Cumulative Points')
    ax2.set_title('Cumulative Points by Season')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No backtest results to visualize")

## 11. Save Models and Export Results

In [ ]:
# Save trained models
import os

models_dir = 'models/'
os.makedirs(models_dir, exist_ok=True)

# Save position-specific models
position_models.save_models(models_dir)

print(f"Models saved to {models_dir}")

In [ ]:
# Export weekly predictions
if weekly_predictions is not None:
    output_file = f'predictions_gw{pipeline.current_gw}.csv'
    weekly_predictions.to_csv(output_file, index=False)
    print(f"Predictions exported to {output_file}")

# Export backtest summary
if len(summary) > 0:
    summary.to_csv('backtest_summary.csv', index=False)
    print("Backtest summary exported to backtest_summary.csv")

## Summary

This notebook implements a comprehensive FPL prediction system with:

### Models
- **Position-specific models** (GK, DEF, MID, FWD) with tailored features
- **Stacking ensemble** combining Random Forest, Gradient Boosting, Ridge, XGBoost, LightGBM
- **Hyperparameter tuning** via GridSearchCV and RandomizedSearchCV

### Features
- **Defensive statistics**: tackles, blocks, interceptions, clearances
- **Fixture difficulty**: FDR from FPL API
- **Historical opponent performance**: performance vs specific teams
- **Rolling averages**: 3-week and 5-week rolling stats

### Strategy Tools
- **Weekly predictions pipeline** with live API data
- **Captain selection model** with differential analysis
- **Chip strategy optimizer**: Wildcard, Bench Boost, Triple Captain, Free Hit timing
- **Historical backtester**: Simulate strategies across past seasons

### Usage
1. Train models: `position_models.train_all_positions(df)`
2. Generate predictions: `pipeline.generate_predictions()`
3. Get captain picks: `captain_selector.get_captain_recommendations(team_ids, predictions)`
4. Optimize chips: `chip_optimizer.find_best_wildcard_timing(teams_df, current_gw)`
5. Backtest: `backtester.backtest_season('2023-24')`